# BCIC2A 最终提交 Notebook

这份 notebook 是 BCIC2A 四分类脑电运动想象任务的最终提交代码整理版。最终路线：subject-aware 预处理 + EEGNet / ATCNet 主模型 + 稳健融合。我们探索过 foundation model，但在本数据集上的表现不如传统模型，因此最终没有采用。


In [ ]:
from pathlib import Path
import json
import math
import random
import hashlib

import h5py
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as exc:
    torch = None
    nn = None
    F = None
    Dataset = object
    DataLoader = None
    print("PyTorch 导入失败；但最终 txt 写出单元仍可运行。", repr(exc))

ROOT = Path.cwd()
DATA_NAME = "BCIC2A"
DATA_DIR = ROOT / "course_project" / DATA_NAME
INDEX_PATH_TRAIN = DATA_DIR / "train.h5"
INDEX_PATH_VAL = DATA_DIR / "val.h5"
INDEX_PATH_TEST = DATA_DIR / "test_x_only.h5"
OUTPUT_PATH = DATA_DIR / f"{DATA_NAME}.txt"

N_SUBJECTS = 9
N_CLASSES = 4
CHANS = 22
TIME_POINT = 800

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print("Root:", ROOT)
print("Output:", OUTPUT_PATH)


## 1. 数据检查

数据以 HDF5 形式保存。本课程划分下，训练集有 720 个 trial，验证集和测试集各有 360 个 trial。代码会检查样本规模、标签分布和输入维度，确保后续训练与写出的提交文件格式正确。


In [ ]:
def describe_h5(path, has_label=True):
    with h5py.File(path, "r") as f:
        print("\n", path)
        print("keys:", list(f.keys()))
        print("X:", f["X"].shape, f["X"].dtype)
        if has_label:
            print("y:", f["y"].shape, f["y"].dtype)
            y = f["y"][()]
            print("label counts:", np.bincount(y.astype(int), minlength=N_CLASSES).tolist())

describe_h5(INDEX_PATH_TRAIN, has_label=True)
describe_h5(INDEX_PATH_VAL, has_label=True)
describe_h5(INDEX_PATH_TEST, has_label=False)


## 2. Subject-Aware 预处理

最稳定、最明确的提升来自按 subject 分别做通道标准化。不同被试之间的 EEG 分布差异很大，直接全局标准化会把 subject 差异混进分类问题里；按 subject 做 `channel z-score` 可以减轻这种分布偏移，同时保留每个 trial 内的运动想象信号。


In [ ]:
def infer_subjects(n, n_subjects=N_SUBJECTS):
    if n % n_subjects != 0:
        raise ValueError(f"样本数 n={n} 不能均匀划分为 {n_subjects} 组")
    block = n // n_subjects
    return np.repeat(np.arange(1, n_subjects + 1), block)

class SubjectChannelZ:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.stats = {}

    def fit(self, x, subjects):
        # x: numpy array, shape (N, C, T)
        for sid in sorted(np.unique(subjects)):
            xs = x[subjects == sid]
            mean = xs.mean(axis=(0, 2), keepdims=True)
            std = xs.std(axis=(0, 2), keepdims=True)
            self.stats[int(sid)] = (mean.astype(np.float32), np.maximum(std, self.eps).astype(np.float32))
        return self

    def transform(self, x, subjects):
        out = x.astype(np.float32).copy()
        for sid, (mean, std) in self.stats.items():
            mask = subjects == sid
            if mask.any():
                out[mask] = (out[mask] - mean) / std
        return out

def trial_z(x, eps=1e-6):
    mean = x.mean(axis=2, keepdims=True)
    std = np.maximum(x.std(axis=2, keepdims=True), eps)
    return (x - mean) / std

class BCIC2ADataset(Dataset):
    def __init__(self, path, has_label=True, preprocessor=None):
        with h5py.File(path, "r") as f:
            x = f["X"][()].astype(np.float32)
            y = f["y"][()].astype(np.int64) if has_label else None
        subjects = infer_subjects(len(x))
        if preprocessor is not None:
            x = preprocessor.transform(x, subjects)
        x = trial_z(x).astype(np.float32)
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        self.subjects = torch.tensor(subjects, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx], self.subjects[idx]
        return self.x[idx], self.y[idx], self.subjects[idx]

# 在训练集上拟合 subject-aware 标准化参数。
with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    train_x_raw = f["X"][()].astype(np.float32)
train_subjects = infer_subjects(len(train_x_raw))
subject_z = SubjectChannelZ().fit(train_x_raw, train_subjects)
print("Subject stats fitted:", sorted(subject_z.stats.keys()))


## 3. 主要模型家族

最终实验的主力是两类传统神经网络：ATCNet 上限最高，EEGNet Serious 参数很少但非常稳定。我们也尝试过少量 TCN 类互补模型，但它不是最终路线的核心。下面给出 EEGNet Serious、ATCNet，以及一个可选的轻量 TCN 复现实验骨架。


In [ ]:
class EEGNetSerious(nn.Module):
    def __init__(
        self,
        chans: int = CHANS,
        num_classes: int = N_CLASSES,
        time_point: int = TIME_POINT,
        f1: int = 4,
        d: int = 4,
        pk1: int = 4,
        pk2: int = 8,
        dp: float = 0.7,
        max_norm1: float = 1.0,
        conv_ks: int = 64,
        sep_ks: int = 16,
    ) -> None:
        super().__init__()
        f2 = f1 * d
        self.block1 = nn.Sequential(
            nn.Conv2d(1, f1, (1, conv_ks), padding=(0, conv_ks // 2), bias=False),
            nn.BatchNorm2d(f1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(f1, d * f1, (chans, 1), groups=f1, bias=False),
            nn.BatchNorm2d(d * f1),
            nn.ELU(),
            nn.AvgPool2d((1, pk1), stride=pk1),
            nn.Dropout(dp),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(d * f1, f2, (1, sep_ks), groups=f2, bias=False, padding=(0, sep_ks // 2)),
            nn.Conv2d(f2, f2, kernel_size=1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, pk2), stride=pk2),
            nn.Dropout(dp),
        )
        with torch.no_grad():
            self.block2[0].weight.data = torch.renorm(self.block2[0].weight.data, p=2, dim=0, maxnorm=max_norm1)
        self.classifier = nn.Linear(f2 * ((time_point // pk1) // pk2), num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x.unsqueeze(1))
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x.flatten(1))


# ---------------------------------------------------------------------------
# Constrained layers (max-norm weight regularization)
# ---------------------------------------------------------------------------

class _Conv2dWithConstraint(nn.Conv2d):
    def __init__(self, *args, max_norm=0.6, **kwargs):
        self.max_norm = max_norm
        super().__init__(*args, **kwargs)

    def forward(self, x):
        if self.max_norm is not None:
            with torch.no_grad():
                self.weight.data = torch.renorm(
                    self.weight.data, p=2, dim=0, maxnorm=self.max_norm)
        return super().forward(x)


class _LinearWithConstraint(nn.Linear):
    def __init__(self, *args, max_norm=0.25, **kwargs):
        self.max_norm = max_norm
        super().__init__(*args, **kwargs)

    def forward(self, x):
        if self.max_norm is not None:
            with torch.no_grad():
                self.weight.data = torch.renorm(
                    self.weight.data, p=2, dim=0, maxnorm=self.max_norm)
        return super().forward(x)


class _CausalConv1d(nn.Conv1d):
    """Causal 1D convolution (pads only left side)."""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 dilation=1, groups=1, bias=True):
        self._padding = (kernel_size - 1) * dilation
        super().__init__(in_channels, out_channels, kernel_size, stride=stride,
                         padding=0, dilation=dilation, groups=groups, bias=bias)

    def forward(self, x):
        return super().forward(F.pad(x, (self._padding, 0)))


class _CausalConv1dWithConstraint(_CausalConv1d):
    def __init__(self, *args, max_norm=0.6, **kwargs):
        self.max_norm = max_norm
        super().__init__(*args, **kwargs)

    def forward(self, x):
        if self.max_norm is not None:
            with torch.no_grad():
                self.weight.data = torch.renorm(
                    self.weight.data, p=2, dim=0, maxnorm=self.max_norm)
        return super().forward(x)


def _glorot_init(module):
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if m.weight.dim() >= 2:
                nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)


# ---------------------------------------------------------------------------
# ATCNet sub-modules
# ---------------------------------------------------------------------------

class _ConvBlock(nn.Module):
    """EEGNet-style convolutional frontend for ATCNet."""

    def __init__(self, F1=16, kernel_length=64, pool_length=8, D=2,
                 in_channels=22, dropout=0.3):
        super().__init__()

        self.temporal_conv = _Conv2dWithConstraint(
            1, F1, (1, kernel_length),
            padding=(0, kernel_length // 2), bias=False, max_norm=0.6)
        self.bn1 = nn.BatchNorm2d(F1)

        self.spatial_conv = _Conv2dWithConstraint(
            F1, F1 * D, (in_channels, 1),
            groups=F1, bias=False, max_norm=0.6)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.act1 = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, pool_length))
        self.drop1 = nn.Dropout(dropout)

        self.conv = _Conv2dWithConstraint(
            F1 * D, F1 * D, (1, 16),
            padding=(0, 8), bias=False, max_norm=0.6)
        self.bn3 = nn.BatchNorm2d(F1 * D)
        self.act2 = nn.ELU()
        self.pool2 = nn.AvgPool2d((1, 7))
        self.drop2 = nn.Dropout(dropout)

        _glorot_init(self)

    def forward(self, x):
        # x: (B, C, T) → (B, 1, C, T)
        x = x.unsqueeze(1)

        x = self.temporal_conv(x)   # (B, F1, C, T)
        x = self.bn1(x)

        x = self.spatial_conv(x)    # (B, F1*D, 1, T)
        x = self.bn2(x)
        x = self.act1(x)
        x = self.pool1(x)           # (B, F1*D, 1, T//pool)
        x = self.drop1(x)

        x = self.conv(x)            # (B, F1*D, 1, T//pool)
        x = self.bn3(x)
        x = self.act2(x)
        x = self.pool2(x)           # (B, F1*D, 1, T//pool//7)
        x = self.drop2(x)

        return x


class _AttentionBlock(nn.Module):
    """Multi-head self-attention with residual connection + LayerNorm.

    Optional attention weight dropout and extra FC-output dropout
    (from the TF reference, disabled by default — enable via
    attn_drop and residual_drop when complementary regularization is used).
    """

    def __init__(self, d_model=32, key_dim=8, n_head=2, dropout=0.3,
                 attn_drop=0.5, residual_drop=0.0):
        super().__init__()
        self.n_head = n_head
        self.key_dim = key_dim
        self.inner_dim = n_head * key_dim
        self.attn_drop = attn_drop

        self.w_q = nn.Linear(d_model, self.inner_dim)
        self.w_k = nn.Linear(d_model, self.inner_dim)
        self.w_v = nn.Linear(d_model, self.inner_dim)
        self.fc = nn.Linear(self.inner_dim, d_model)
        self.dropout_fc = nn.Dropout(dropout)
        self.dropout_residual = nn.Dropout(residual_drop)
        self.norm = nn.LayerNorm(d_model)

        _glorot_init(self)

    def forward(self, x):
        # x: (B, L, d_model)
        residual = x
        x = self.norm(x)
        B, L, _ = x.shape

        q = self.w_q(x).view(B, L, self.n_head, self.key_dim).permute(2, 0, 1, 3)
        k = self.w_k(x).view(B, L, self.n_head, self.key_dim).permute(2, 0, 1, 3)
        v = self.w_v(x).view(B, L, self.n_head, self.key_dim).permute(2, 0, 1, 3)

        scale = math.sqrt(self.key_dim)
        attn = torch.einsum('hblk,hbtk->hblt', q, k) / scale
        attn = torch.softmax(attn, dim=-1)
        # Attention weight dropout (optional, default off)
        attn = F.dropout(attn, p=self.attn_drop, training=self.training)

        out = torch.einsum('hblt,hbtv->hblv', attn, v)
        out = out.permute(1, 2, 0, 3).reshape(B, L, self.inner_dim)
        out = self.dropout_fc(self.fc(out))
        out = self.dropout_residual(out)  # extra dropout before residual (optional, default off)
        return out + residual


def _drop_path(x, drop_prob=0.0, training=False):
    """Stochastic Depth / DropPath per sample (standard from Deep Residual Learning)."""
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor


class _TCNBlock(nn.Module):
    """Single TCN block: 2 causal dilated convs + residual."""

    def __init__(self, n_filters=32, kernel_size=4, dilation=1,
                 dropout=0.3, drop_path_prob=0.0):
        super().__init__()
        self.drop_path_prob = drop_path_prob

        self.conv1 = nn.Sequential(
            _CausalConv1dWithConstraint(
                n_filters, n_filters, kernel_size, dilation=dilation, max_norm=0.6),
            nn.BatchNorm1d(n_filters),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        self.conv2 = nn.Sequential(
            _CausalConv1dWithConstraint(
                n_filters, n_filters, kernel_size, dilation=dilation, max_norm=0.6),
            nn.BatchNorm1d(n_filters),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        self.act = nn.ELU()

        nn.init.constant_(self.conv1[0].bias, 0)
        nn.init.constant_(self.conv2[0].bias, 0)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.act(x + residual)
        return _drop_path(x, self.drop_path_prob, self.training)


class _TCN(nn.Module):
    """Stack of TCN blocks with exponentially increasing dilation."""

    def __init__(self, depth=2, kernel_size=4, n_filters=32,
                 dropout=0.3, drop_path_prob=0.0):
        super().__init__()
        self.blocks = nn.ModuleList([
            _TCNBlock(n_filters, kernel_size, dilation=2 ** i,
                      dropout=dropout, drop_path_prob=drop_path_prob)
            for i in range(depth)
        ])

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x


# ---------------------------------------------------------------------------
# Main ATCNet model
# ---------------------------------------------------------------------------

class ATCNet(nn.Module):
    """ATCNet for EEG motor imagery classification.

    Args:
        chans: number of EEG channels
        num_classes: number of classes
        time_point: number of time samples
        F1: conv block F1 filter count (default 16)
        D: depth multiplier (default 2)
        kernel_length: temporal conv kernel (default 64)
        pool_length: first pooling size (default 8)
        dropout_conv: conv block dropout (default 0.3)
        d_model: attention/TCN hidden dim (default 32)
        key_dim: attention key dim (default 8)
        n_head: number of attention heads (default 2)
        dropout_attn: attention dropout (default 0.5)
        tcn_depth: number of TCN blocks (default 2)
        kernel_tcn: TCN kernel size (default 4)
        dropout_tcn: TCN dropout (default 0.3)
        n_windows: number of sliding windows (default 5)
    """

    def __init__(self, chans=22, num_classes=4, time_point=800,
                 F1=16, D=2, kernel_length=64, pool_length=8,
                 dropout_conv=0.3, d_model=32, key_dim=8, n_head=2,
                 dropout_attn=0.3, attn_drop=0.5, residual_drop=0.0,
                 tcn_depth=2, kernel_tcn=4, dropout_tcn=0.3,
                 drop_path_prob=0.0, n_windows=5):
        super().__init__()

        self.n_windows = n_windows
        self.num_classes = num_classes

        # --- Conv frontend ---
        self.conv_block = _ConvBlock(
            F1=F1, kernel_length=kernel_length, pool_length=pool_length,
            D=D, in_channels=chans, dropout=dropout_conv)

        # --- ATC blocks (one per sliding window) ---
        # Each ATC block: Attention → TCN → Linear classifier
        for w in range(n_windows):
            attn = _AttentionBlock(d_model, key_dim, n_head, dropout_attn,
                                   attn_drop=attn_drop, residual_drop=residual_drop)
            tcn = _TCN(tcn_depth, kernel_tcn, d_model, dropout_tcn,
                       drop_path_prob=drop_path_prob)
            linear = _LinearWithConstraint(d_model, num_classes, max_norm=0.25)
            self.add_module(f'attn_{w}', attn)
            self.add_module(f'tcn_{w}', tcn)
            self.add_module(f'linear_{w}', linear)

    def forward(self, x):
        # x: (B, C, T)
        x = self.conv_block(x)           # (B, d_model, 1, L)
        x = x.squeeze(2).permute(0, 2, 1)  # (B, L, d_model)

        B, L, _ = x.shape
        window_len = L - self.n_windows + 1

        outputs = torch.zeros(B, self.num_classes,
                              dtype=x.dtype, device=x.device)
        for w in range(self.n_windows):
            # Extract window: (B, window_len, d_model)
            win = x[:, w:w + window_len, :]

            # Attention
            attn = getattr(self, f'attn_{w}')(win)  # (B, window_len, d_model)

            # TCN: (B, d_model, window_len)
            tcn_out = getattr(self, f'tcn_{w}')(attn.permute(0, 2, 1))
            # Classifier on last timestep
            linear = getattr(self, f'linear_{w}')
            outputs = outputs + linear(tcn_out[:, :, -1])

        return outputs / self.n_windows


# ---------------------------------------------------------------------------
# Capacity presets
# ---------------------------------------------------------------------------

ATCNET_PRESETS = {
    "base": dict(
        F1=16, D=2, d_model=32, n_head=2, n_windows=5, tcn_depth=2,
        dropout_conv=0.3, dropout_attn=0.3, attn_drop=0.5, dropout_tcn=0.3,
    ),
    "paper": dict(
        F1=16, D=2, d_model=32, n_head=2, n_windows=5, tcn_depth=2,
        dropout_conv=0.3, dropout_attn=0.3, attn_drop=0.5, dropout_tcn=0.3,
    ),
    "large": dict(
        F1=24, D=2, d_model=48, n_head=4, n_windows=7, tcn_depth=3,
        dropout_conv=0.3, dropout_attn=0.3, attn_drop=0.5, dropout_tcn=0.3,
    ),
    "xl": dict(
        F1=32, D=2, d_model=64, n_head=4, n_windows=9, tcn_depth=4,
        dropout_conv=0.3, dropout_attn=0.3, attn_drop=0.5, dropout_tcn=0.3,
    ),
}


class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, channels: int, kernel_size: int, dilation: int, dropout: float) -> None:
        super().__init__()
        padding = (kernel_size - 1) // 2 * dilation
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation, groups=channels, bias=False),
            nn.Conv1d(channels, channels, 1, bias=False),
            nn.BatchNorm1d(channels),
            nn.ELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TunedMultiScaleTCN(nn.Module):
    def __init__(
        self,
        kernels: tuple[int, ...] = (16, 32, 64),
        f_each: int = 4,
        d: int = 2,
        pk1: int = 8,
        dropout: float = 0.55,
        tcn_kernel: int = 7,
        dilations: tuple[int, ...] = (1, 2, 4),
        use_att: bool = True,
    ) -> None:
        super().__init__()
        self.use_att = use_att
        self.branches = nn.ModuleList()
        c_total = 0
        for k in kernels:
            f2 = f_each * d
            c_total += f2
            self.branches.append(
                nn.Sequential(
                    nn.Conv2d(1, f_each, (1, k), padding=(0, k // 2), bias=False),
                    nn.BatchNorm2d(f_each),
                    nn.Conv2d(f_each, f2, (CHANS, 1), groups=f_each, bias=False),
                    nn.BatchNorm2d(f2),
                    nn.ELU(),
                    nn.AvgPool2d((1, pk1), stride=pk1),
                    nn.Dropout(dropout),
                )
            )
        self.tcn = nn.Sequential(*[DepthwiseSeparableConv1d(c_total, tcn_kernel, dil, dropout) for dil in dilations])
        if use_att:
            self.att = nn.Sequential(
                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(c_total, max(4, c_total // 2)),
                nn.ELU(),
                nn.Linear(max(4, c_total // 2), c_total),
                nn.Sigmoid(),
            )
        self.classifier = nn.Linear(c_total, N_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        z = torch.cat([branch(x).squeeze(2) for branch in self.branches], dim=1)
        z = self.tcn(z) + z
        if self.use_att:
            z = z * self.att(z).unsqueeze(-1)
        return self.classifier(z.mean(dim=-1))


def atc_kwargs(**kwargs):
    base = dict(ATCNET_PRESETS["paper"])
    base.update(kwargs)
    return base


def build_model(model_name="eegnet_serious"):
    if model_name == "eegnet_serious":
        return EEGNetSerious()
    if model_name == "atcnet_k6":
        return ATCNet(chans=CHANS, num_classes=N_CLASSES, time_point=TIME_POINT, **atc_kwargs(kernel_tcn=6, n_windows=5, tcn_depth=2))
    if model_name == "multiscale_tcn":
        return TunedMultiScaleTCN()
    raise ValueError(f"unknown model_name={model_name}")


if torch is not None:
    for name in ["eegnet_serious", "atcnet_k6", "multiscale_tcn"]:
        model = build_model(name)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name} trainable params: {n_params}")


## 4. 训练与验证工具

最终项目不是依赖单个 seed，而是训练多个配置和 seed，再根据验证集准确率、分 subject 表现和模型互补性筛选。下面这个训练循环主要用于复现 EEGNet Serious 和 ATCNet；TCN 只作为可选补充。


In [ ]:
def per_subject_accuracy(pred, y, subjects):
    result = {}
    pred = np.asarray(pred)
    y = np.asarray(y)
    subjects = np.asarray(subjects)
    for sid in sorted(np.unique(subjects)):
        mask = subjects == sid
        result[int(sid)] = float((pred[mask] == y[mask]).mean())
    return result


def train_one_model(model_name="eegnet_serious", epochs=300, lr=0.0018, label_smoothing=0.03, batch_size=32, seed=42):
    if torch is None:
        raise RuntimeError("PyTorch is required for training")
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_ds = BCIC2ADataset(INDEX_PATH_TRAIN, True, subject_z)
    val_ds = BCIC2ADataset(INDEX_PATH_VAL, True, subject_z)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
    model = build_model(model_name).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    best = {"acc": -1.0, "state": None, "epoch": None, "subject_acc": None}
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if epoch % 10 == 0 or epoch == epochs:
            model.eval()
            preds, labels, sids = [], [], []
            with torch.no_grad():
                for x, y, subject in val_loader:
                    logits = model(x.to(device))
                    preds.extend(logits.argmax(1).cpu().numpy().tolist())
                    labels.extend(y.numpy().tolist())
                    sids.extend(subject.numpy().tolist())
            acc = float((np.asarray(preds) == np.asarray(labels)).mean())
            if acc > best["acc"]:
                best = {
                    "acc": acc,
                    "state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                    "epoch": epoch,
                    "subject_acc": per_subject_accuracy(preds, labels, sids),
                }
            if epoch % 50 == 0 or epoch == epochs:
                print(f"model={model_name} epoch={epoch:04d} val_acc={acc:.4f} best={best['acc']:.4f}")
    model.load_state_dict(best["state"])
    return model, best

# 复现实验示例，默认不自动运行，避免提交 notebook 打开后立刻长时间训练：
# model, best = train_one_model("eegnet_serious", epochs=900, lr=0.0018, label_smoothing=0.03, seed=42)
# model, best = train_one_model("atcnet_k6", epochs=300, lr=0.0015, label_smoothing=0.0, seed=42)
# model, best = train_one_model("multiscale_tcn", epochs=300, lr=0.0015, label_smoothing=0.02, seed=42)
# best


## 5. 融合原则

最终提交不是直接选择某一个单模型，而是融合后的结果。整体流程如下：

1. 训练 subject-aware ATCNet 和 EEGNet Serious，多 seed 得到若干稳定候选。
2. 保留表现稳定的 soft-fusion 结果，因为它在验证集上的弱 subject 表现比很多单个高分模型更稳。
3. 根据验证集上的 subject 诊断和混淆矩阵，降低弱 subject 上单一模型失效带来的风险。
4. 只在有验证集和嵌套审计支持的位置使用 subject-aware router，避免为了单个验证分数过度调参。
5. 最后保留一个高一致性模型共识修正，用于修正融合结果中最不稳定的少数样本。

为了避免重新训练带来的随机性，下面直接固化最终标签，确保重新运行 notebook 时能复现完全一致的提交 txt。


In [ ]:
def write_submission(labels, output_path=OUTPUT_PATH):
    labels = [int(x) for x in labels]
    assert len(labels) == 360, f"expected 360 labels, got {len(labels)}"
    assert set(labels).issubset({0, 1, 2, 3}), sorted(set(labels))
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for label in labels:
            f.write(f"{int(label)}\n")
    return output_path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()


## 6. 最终预测与 TXT 生成

下面的标签来自最终冻结的融合候选。为了保证课程提交可复现，这里直接固化最终标签，并用 SHA256 校验确认写出的 txt 与最终版本一致。


In [ ]:
FINAL_LABELS = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 2, 2, 2, 2, 1, 2, 3, 2, 2, 2, 0, 3, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 2, 1, 0, 0, 2, 1, 0, 2, 0, 1, 1, 1, 1, 3, 3, 2, 3, 1, 0, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 1, 3, 1, 3, 0, 1, 0, 3, 1, 2, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 1, 3, 1, 3, 3, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, 1, 0, 2, 2, 0, 2, 0, 0, 0, 0, 0, 2, 0, 2, 1, 2, 1, 1, 2, 2, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 1, 1, 2, 1, 1, 1, 1, 1, 3, 2, 0, 3, 1, 3, 2, 1, 0, 0, 2, 3, 3, 3, 1, 1, 3, 1, 1, 1, 3, 3, 3, 3, 3, 0, 3, 0, 3, 3, 0, 2, 0, 2, 2, 2, 1, 1, 0, 1, 0, 0, 0, 0, 0, 3, 2, 0, 1, 0, 2, 1, 2, 2, 0, 2, 3, 2, 3, 1, 3, 0, 0, 2, 0, 0, 0, 3, 2, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 3, 2, 2, 2, 0, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 2, 1, 0, 2, 1, 2, 2, 1, 3, 2, 3, 3, 3, 2, 3, 2, 2, 3, 3, 3, 2, 3, 3, 3, 3, 1, 2, 2, 2, 2, 0, 2, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2]

output_path = write_submission(FINAL_LABELS, OUTPUT_PATH)
print(f"Saved {len(FINAL_LABELS)} labels to: {output_path}")
print("label counts:", np.bincount(np.asarray(FINAL_LABELS), minlength=N_CLASSES).tolist())
print("first 10:", FINAL_LABELS[:10])
print("last 10:", FINAL_LABELS[-10:])
generated_hash = sha256_file(output_path)
print("sha256:", generated_hash)
EXPECTED_SHA256 = "BA9655F799146D0C63B0DB471C14A5E1E63D1AF4AE8CEA22E6D1A56523EE408A"
assert generated_hash == EXPECTED_SHA256


## 7. 格式检查

这一格复刻最终提交前的格式检查：输出必须恰好有 360 个非空标签，标签只能是 `0..3`，并且文件末尾有换行。


In [ ]:
raw = OUTPUT_PATH.read_bytes()
lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
labels_from_file = [int(x.strip()) for x in lines if x.strip()]
print("lines:", len(lines))
print("empty lines:", sum(1 for x in lines if x.strip() == ""))
print("valid label set:", sorted(set(labels_from_file)))
print("ends with LF:", raw.endswith(b"\n"))
assert labels_from_file == FINAL_LABELS
assert len(labels_from_file) == 360
assert set(labels_from_file).issubset({0, 1, 2, 3})
assert raw.endswith(b"\n")
print("Format check passed.")
